# Week 2 Lab: Data Validation & Quality

**CS 203: Software Tools and Techniques for AI**

---

## Lab Overview

In this lab, you will learn to validate and clean data using:
1. **Unix CLI tools** - Quick data inspection
2. **jq** - JSON processing
3. **pandas** - Data profiling and cleaning
4. **Pydantic** - Schema validation
5. **Great Expectations** - Data quality testing

**Goal**: Transform messy movie data into clean, validated data ready for ML.

---

## Setup

In [ ]:
# Install required packages (run this first!)
!sudo apt-get install -y jq -qq  # JSON processor for CLI
!pip install -q pandas pydantic csvkit

# Verify installations
!echo "jq version:" && jq --version
!echo "csvkit version:" && csvlook --version 2>&1 | head -1

In [ ]:
import pandas as pd
import numpy as np
import json
from typing import Optional, List
from pydantic import BaseModel, Field, validator, ValidationError

print("All imports successful!")

---

# Part 1: Sample Messy Data

Let's create some realistic messy movie data that mimics what you'd get from APIs.

In [ ]:
# Create sample messy data (simulating what we collected in Week 1)
messy_movies = [
    {"Title": "Inception", "Year": "2010", "Runtime": "148 min", "imdbRating": "8.8", "BoxOffice": "$292,576,195", "Genre": "Action, Sci-Fi"},
    {"Title": "Avatar", "Year": "2009", "Runtime": "162 min", "imdbRating": "7.9", "BoxOffice": "$760,507,625", "Genre": "Action, Adventure"},
    {"Title": "The Room", "Year": "2003", "Runtime": "99 min", "imdbRating": "3.9", "BoxOffice": "N/A", "Genre": "Drama"},
    {"Title": "Inception", "Year": "2010", "Runtime": "148 min", "imdbRating": "8.8", "BoxOffice": "$292,576,195", "Genre": "Action, Sci-Fi"},  # Duplicate!
    {"Title": "Tenet", "Year": "N/A", "Runtime": "150 min", "imdbRating": "7.3", "BoxOffice": "N/A", "Genre": "Action, Sci-Fi"},  # Missing year
    {"Title": "The Matrix", "Year": "1999", "Runtime": "136 min", "imdbRating": "8.7", "BoxOffice": "$171,479,930", "Genre": "Action, Sci-Fi"},
    {"Title": "Interstellar", "Year": "2014", "Runtime": "", "imdbRating": "8.6", "BoxOffice": "$188,020,017", "Genre": "Adventure, Drama"},  # Empty runtime
    {"Title": "", "Year": "2020", "Runtime": "120 min", "imdbRating": "7.0", "BoxOffice": "$50,000,000", "Genre": "Comedy"},  # Missing title!
    {"Title": "Joker", "Year": "2019", "Runtime": "122 min", "imdbRating": "invalid", "BoxOffice": "$335,451,311", "Genre": "Crime, Drama"},  # Invalid rating
    {"Title": "Parasite", "Year": "2019", "Runtime": "132 min", "imdbRating": "8.5", "BoxOffice": "$53,369,749", "Genre": "Drama, Thriller"},
]

# Save as JSON for CLI exercises
with open('messy_movies.json', 'w') as f:
    json.dump(messy_movies, f, indent=2)

# Create DataFrame
df_messy = pd.DataFrame(messy_movies)
df_messy.to_csv('messy_movies.csv', index=False)

print("Created messy_movies.json and messy_movies.csv")
print(f"\nDataFrame shape: {df_messy.shape}")
df_messy

---

# Part 2: CLI Data Inspection

Before writing code, always inspect your data with CLI tools.

### Question 2.1 (Solved): Basic File Inspection

In [ ]:
# SOLVED EXAMPLE
# Check file sizes and line counts
!echo "=== File sizes ==="
!ls -lh messy_movies.json messy_movies.csv

!echo "\n=== Line counts ==="
!wc -l messy_movies.json messy_movies.csv

!echo "\n=== First 5 lines of CSV ==="
!head -5 messy_movies.csv

### Question 2.2: Inspect JSON with jq

Use `jq` to:
1. Pretty-print the JSON file
2. Get the length (number of movies)
3. Extract just the titles

In [ ]:
# YOUR CODE HERE
# Pretty print
!jq '.' messy_movies.json | head -40

# Get length (number of movies)
!echo "\nTotal movies:" && jq 'length' messy_movies.json

# Extract all titles
!echo "\nAll titles:" && jq '.[].Title' messy_movies.json


### Question 2.3: Find Data Issues with jq

Use `jq` to find:
1. Movies where Year is "N/A"
2. Movies where BoxOffice is "N/A"
3. Movies where Title is empty

**Hint**: Use `select()` function in jq

In [ ]:
# YOUR CODE HERE
# 1. Movies where Year is "N/A"
!echo "=== Movies with Year = N/A ===" && jq '[.[] | select(.Year == "N/A")] | length' messy_movies.json

# 2. Movies where BoxOffice is "N/A"
!echo "\n=== Movies with BoxOffice = N/A ===" && jq '[.[] | select(.BoxOffice == "N/A")] | length' messy_movies.json

# 3. Movies with imdbRating below 5
!echo "\n=== Movies with imdbRating < 5 ===" && jq '[.[] | select(.imdbRating | tonumber < 5)] | .[].Title' messy_movies.json


---

# Part 3: Data Profiling with Pandas

Now let's do systematic data profiling.

### Question 3.1 (Solved): Basic Data Profiling

In [ ]:
# SOLVED EXAMPLE
print("=== Data Types ===")
print(df_messy.dtypes)

print("\n=== Missing Values ===")
print(df_messy.isnull().sum())

print("\n=== Unique Values per Column ===")
print(df_messy.nunique())

print("\n=== Sample Values ===")
for col in df_messy.columns:
    print(f"{col}: {df_messy[col].unique()[:5]}")

### Question 3.2: Identify All Data Quality Issues

Write a function `profile_data(df)` that returns a dictionary summarizing:
1. Total rows
2. Duplicate rows
3. Missing values per column (including "N/A" strings)
4. Empty strings per column
5. Data type issues (strings that should be numbers)

In [ ]:
# YOUR CODE HERE
def profile_data(df):
    """Generate a data quality profile."""
    profile = {}

    for col in df.columns:
        col_data = df[col]
        profile[col] = {
            "dtype":          str(col_data.dtype),
            "n_missing":      int(col_data.isna().sum()),
            "pct_missing":    round(col_data.isna().mean() * 100, 2),
            "n_unique":       int(col_data.nunique()),
            "n_na_string":    int((col_data == "N/A").sum()) if col_data.dtype == object else 0,
            "sample_values":  col_data.dropna().unique()[:3].tolist(),
        }

    profile["_summary"] = {
        "n_rows":       len(df),
        "n_cols":       len(df.columns),
        "total_missing": int(df.isna().sum().sum()),
        "n_duplicates": int(df.duplicated().sum()),
    }
    return profile

# Test
profile = profile_data(df_messy)
print(json.dumps(profile, indent=2, default=str))


### Question 3.3: Find Duplicates

Find all duplicate rows in the dataset. How many duplicates are there? Which movies are duplicated?

In [ ]:
# YOUR CODE HERE
# Find duplicate rows
duplicates = df_messy[df_messy.duplicated(keep=False)]
n_dups = df_messy.duplicated().sum()

print(f"Total duplicate rows   : {n_dups}")
print(f"Unique duplicated movies: {duplicates['Title'].nunique()}")
print("\nDuplicated titles:")
print(duplicates['Title'].value_counts())
print("\nDuplicate rows:")
print(duplicates[['Title', 'Year', 'imdbRating']].to_string(index=True))


---

# Part 4: Data Cleaning

Now let's clean the data systematically.

### Question 4.1 (Solved): Clean Runtime Column

In [ ]:
# SOLVED EXAMPLE
def clean_runtime(runtime_str):
    """Convert '148 min' to integer 148."""
    if pd.isna(runtime_str) or runtime_str == '' or runtime_str == 'N/A':
        return None
    # Extract digits
    import re
    match = re.search(r'(\d+)', str(runtime_str))
    if match:
        return int(match.group(1))
    return None

# Test
print(clean_runtime('148 min'))  # 148
print(clean_runtime('N/A'))      # None
print(clean_runtime(''))         # None

### Question 4.2: Clean BoxOffice Column

Write a function `clean_box_office(value)` that converts:
- `"$292,576,195"` → `292576195` (integer)
- `"N/A"` → `None`
- `""` → `None`

In [ ]:
# YOUR CODE HERE
import re

def clean_box_office(value):
    """Convert '$292,576,195' to integer."""
    if not value or str(value).strip() in ('N/A', 'n/a', '', 'None'):
        return None
    # Remove $ and commas, then parse
    cleaned = re.sub(r'[^0-9]', '', str(value))
    return int(cleaned) if cleaned else None

# Test
print(clean_box_office('$292,576,195'))  # 292576195
print(clean_box_office('N/A'))           # None
print(clean_box_office('$1,000'))        # 1000
print(clean_box_office(''))             # None


### Question 4.3: Clean Year Column

Write a function `clean_year(value)` that:
- Converts valid year strings to integers
- Returns `None` for "N/A" or invalid values
- Validates that year is between 1888 (first film) and current year + 2

In [ ]:
# YOUR CODE HERE
def clean_year(value):
    """Convert year strings to int; return None for invalid."""
    import re
    if not value or str(value).strip() in ('N/A', '', 'None'):
        return None
    match = re.search(r'(\d{4})', str(value))
    if match:
        year = int(match.group(1))
        return year if 1888 <= year <= 2030 else None
    return None

# Test
print(clean_year('2010'))    # 2010
print(clean_year('N/A'))     # None
print(clean_year(''))        # None
print(clean_year('2010–'))   # 2010
print(clean_year('1800'))    # None (too old)


### Question 4.4: Clean Rating Column

Write a function `clean_rating(value)` that:
- Converts valid rating strings to floats
- Returns `None` for invalid values
- Validates that rating is between 0.0 and 10.0

In [ ]:
# YOUR CODE HERE
def clean_rating(value):
    """Convert rating strings to float in [0, 10]; return None for invalid."""
    if not value or str(value).strip() in ('N/A', '', 'None'):
        return None
    try:
        rating = float(str(value).strip())
        return rating if 0.0 <= rating <= 10.0 else None
    except ValueError:
        return None

# Test
print(clean_rating('8.8'))   # 8.8
print(clean_rating('N/A'))   # None
print(clean_rating(''))      # None
print(clean_rating('11.0'))  # None (out of range)
print(clean_rating('0.0'))   # 0.0


### Question 4.5: Complete Cleaning Pipeline

Create a function `clean_movie_data(df)` that:
1. Removes duplicates
2. Removes rows with empty titles
3. Cleans all columns using the functions above
4. Returns a clean DataFrame with proper data types

In [ ]:
# YOUR CODE HERE
def clean_movie_data(df):
    """Complete cleaning pipeline."""
    import re
    df = df.copy()

    # 1. Remove exact duplicates
    df = df.drop_duplicates()

    # 2. Apply column cleaners
    if 'BoxOffice' in df.columns:
        df['BoxOffice'] = df['BoxOffice'].apply(clean_box_office)
    if 'Year' in df.columns:
        df['Year'] = df['Year'].apply(clean_year)
    if 'imdbRating' in df.columns:
        df['imdbRating'] = df['imdbRating'].apply(clean_rating)
    if 'Runtime' in df.columns:
        df['Runtime'] = df['Runtime'].apply(clean_runtime)

    # 3. Remove rows with empty/null Title
    df = df[df['Title'].notna() & (df['Title'].str.strip() != '')]

    # 4. Reset index
    df = df.reset_index(drop=True)
    return df

# Test
df_clean = clean_movie_data(df_messy)
print(f"Before: {len(df_messy)} rows")
print(f"After:  {len(df_clean)} rows")
print(df_clean[['Title','Year','imdbRating','BoxOffice']].to_string(index=False))


---

# Part 5: Schema Validation with Pydantic

Pydantic provides type validation and data parsing.

### Question 5.1 (Solved): Define a Movie Schema

In [ ]:
# SOLVED EXAMPLE
from pydantic import BaseModel, Field, field_validator
from typing import Optional
import re

class Movie(BaseModel):
    """Validated movie schema."""
    title: str = Field(..., min_length=1, description="Movie title")
    year: int = Field(..., ge=1888, le=2030, description="Release year")
    runtime_minutes: Optional[int] = Field(None, ge=1, le=1000)
    imdb_rating: Optional[float] = Field(None, ge=0, le=10)
    box_office: Optional[int] = Field(None, ge=0)
    genre: str = Field(..., min_length=1)
    
    @field_validator('title')
    @classmethod
    def title_not_empty(cls, v):
        if not v or not v.strip():
            raise ValueError('Title cannot be empty')
        return v.strip()

# Test with valid data
movie = Movie(
    title="Inception",
    year=2010,
    runtime_minutes=148,
    imdb_rating=8.8,
    box_office=292576195,
    genre="Action, Sci-Fi"
)
print("Valid movie:")
print(movie.model_dump())

### Question 5.2: Test Validation Errors

Try creating Movie objects with invalid data and observe the validation errors:
1. Empty title
2. Year before 1888
3. Rating above 10
4. Negative box office

In [ ]:
# YOUR CODE HERE
# Try each invalid case and catch ValidationError

invalid_cases = [
    {"name": "Year too old",    "data": {"title": "Old Film",   "year": 1800,  "genre": "Drama"}},
    {"name": "Empty title",     "data": {"title": "",            "year": 2010,  "genre": "Action"}},
    {"name": "Rating too high", "data": {"title": "Good Movie", "year": 2020,  "genre": "Comedy",  "imdb_rating": 11.0}},
    {"name": "Negative runtime","data": {"title": "Short Film",  "year": 2015,  "genre": "Short",   "runtime_minutes": -5}},
]

for case in invalid_cases:
    try:
        Movie(**case["data"])
        print(f"✗ {case['name']}: No error raised (unexpected!)")
    except Exception as e:
        print(f"✓ {case['name']}: {type(e).__name__} — {str(e)[:100]}")


### Question 5.3: Validate and Convert Raw Data

Write a function `validate_movies(raw_data)` that:
1. Takes a list of raw movie dictionaries (from JSON)
2. Attempts to clean and validate each movie
3. Returns two lists: `valid_movies` and `invalid_movies` (with error messages)

In [ ]:
# YOUR CODE HERE
def validate_movies(raw_data):
    """Validate a list of raw movie dictionaries."""
    valid_movies   = []
    invalid_movies = []

    for raw in raw_data:
        try:
            movie = Movie(
                title            = raw.get("Title", ""),
                year             = clean_year(raw.get("Year")) or 0,
                genre            = raw.get("Genre", "Unknown"),
                runtime_minutes  = clean_runtime(raw.get("Runtime")),
                imdb_rating      = clean_rating(raw.get("imdbRating")),
                box_office       = clean_box_office(raw.get("BoxOffice")),
            )
            valid_movies.append(movie.model_dump())
        except Exception as e:
            invalid_movies.append({"raw": raw, "error": str(e)})

    return valid_movies, invalid_movies

# Test
valid, invalid = validate_movies(messy_movies)
print(f"Valid: {len(valid)}, Invalid: {len(invalid)}")
for m in valid[:3]:
    print(f"  {m['title']} ({m['year']})")
for m in invalid[:2]:
    print(f"  INVALID: {m['raw'].get('Title')} — {m['error'][:80]}")


---

# Part 6: Data Quality Assertions

Write assertions that should pass for clean data.

### Question 6.1 (Solved): Data Quality Checks

In [ ]:
# SOLVED EXAMPLE
def check_data_quality(df):
    """Run data quality assertions."""
    checks = []
    
    # Check 1: No duplicate rows
    duplicates = df.duplicated().sum()
    checks.append(("No duplicates", duplicates == 0, f"Found {duplicates} duplicates"))
    
    # Check 2: No empty titles
    empty_titles = (df['Title'] == '').sum() + df['Title'].isna().sum()
    checks.append(("No empty titles", empty_titles == 0, f"Found {empty_titles} empty titles"))
    
    # Check 3: Year in valid range
    if 'Year' in df.columns:
        invalid_years = ((df['Year'] < 1888) | (df['Year'] > 2030)).sum()
        checks.append(("Valid years", invalid_years == 0, f"Found {invalid_years} invalid years"))
    
    # Print results
    print("Data Quality Checks:")
    print("-" * 50)
    for name, passed, message in checks:
        status = "✓" if passed else "✗"
        print(f"{status} {name}: {message if not passed else 'OK'}")
    
    return all(passed for _, passed, _ in checks)

# Test on messy data (should fail)
print("Checking messy data:")
check_data_quality(df_messy)

### Question 6.2: Add More Quality Checks

Extend the `check_data_quality` function to include:
1. Rating values between 0 and 10
2. No negative box office values
3. Runtime between 1 and 1000 minutes
4. At least 90% of rows have non-null ratings

In [ ]:
# YOUR CODE HERE
def check_data_quality_extended(df):
    """Extended data quality checks."""
    checks = []

    # -- original checks (copied) --
    duplicates = df.duplicated().sum()
    checks.append(("No duplicates",   duplicates == 0,       f"Found {duplicates} duplicates"))
    empty_titles = (df['Title'] == '').sum() + df['Title'].isna().sum()
    checks.append(("No empty titles", empty_titles == 0,     f"Found {empty_titles} empty titles"))

    # 1. Rating values between 0 and 10
    if 'imdbRating' in df.columns:
        numeric_ratings = pd.to_numeric(df['imdbRating'], errors='coerce').dropna()
        bad_ratings = ((numeric_ratings < 0) | (numeric_ratings > 10)).sum()
        checks.append(("Ratings in [0,10]", bad_ratings == 0, f"{bad_ratings} ratings out of range"))

    # 2. No negative box office values
    if 'BoxOffice' in df.columns:
        numeric_bo = pd.to_numeric(df['BoxOffice'], errors='coerce').dropna()
        neg_bo = (numeric_bo < 0).sum()
        checks.append(("No negative box office", neg_bo == 0, f"{neg_bo} negative box office values"))

    # 3. Runtime between 1 and 1000 minutes
    if 'Runtime' in df.columns:
        numeric_rt = pd.to_numeric(df['Runtime'], errors='coerce').dropna()
        bad_rt = ((numeric_rt < 1) | (numeric_rt > 1000)).sum()
        checks.append(("Runtime in [1,1000]", bad_rt == 0, f"{bad_rt} invalid runtimes"))

    # 4. At least 90% of rows have non-null ratings
    if 'imdbRating' in df.columns:
        pct_rated = df['imdbRating'].notna().mean()
        checks.append(("≥90% have ratings", pct_rated >= 0.9, f"Only {pct_rated:.1%} rows have ratings"))

    # Print results
    print("Extended Data Quality Checks:")
    print("-" * 50)
    for name, passed, detail in checks:
        status = "✓ PASS" if passed else "✗ FAIL"
        print(f"{status}  {name}: {detail}")

    passed_all = all(c[1] for c in checks)
    print(f"\nOverall: {'All checks passed!' if passed_all else 'Some checks FAILED.'}")
    return checks

# Test on clean data
check_data_quality_extended(df_clean)


---

# Part 7: Complete Pipeline

Put everything together into a complete validation pipeline.

### Question 7.1: Build the Complete Pipeline

Create a `DataValidationPipeline` class that:
1. Loads data from JSON or CSV
2. Profiles the data
3. Cleans the data
4. Validates with Pydantic
5. Runs quality checks
6. Exports clean data

In [ ]:
# YOUR CODE HERE
class DataValidationPipeline:
    """Complete data validation pipeline."""

    def __init__(self, input_file):
        self.input_file = input_file
        self.raw_data   = None
        self.clean_data = None
        self._profile   = {}
        self._valid     = []
        self._invalid   = []

    def load(self):
        """Load data from JSON or CSV file."""
        if self.input_file.endswith('.json'):
            with open(self.input_file) as f:
                self.raw_data = json.load(f)
        else:
            df = pd.read_csv(self.input_file)
            self.raw_data = df.to_dict('records')
        print(f"Loaded {len(self.raw_data)} records from {self.input_file}")
        return self

    def profile(self):
        """Generate data profile."""
        df = pd.DataFrame(self.raw_data)
        self._profile = profile_data(df)
        print(f"Profile: {self._profile['_summary']}")
        return self

    def clean(self):
        """Clean the data."""
        df = pd.DataFrame(self.raw_data)
        self.clean_data = clean_movie_data(df)
        print(f"Cleaned: {len(self.raw_data)} → {len(self.clean_data)} rows")
        return self

    def validate(self):
        """Validate with Pydantic."""
        raw = self.clean_data.to_dict('records') if self.clean_data is not None else self.raw_data
        self._valid, self._invalid = validate_movies(raw)
        print(f"Validated: {len(self._valid)} valid, {len(self._invalid)} invalid")
        return self

    def check_quality(self):
        """Run quality checks."""
        if self.clean_data is not None:
            check_data_quality_extended(self.clean_data)
        return self

    def export(self, output_file):
        """Export clean validated data."""
        out_df = pd.DataFrame(self._valid)
        if output_file.endswith('.csv'):
            out_df.to_csv(output_file, index=False)
        else:
            out_df.to_json(output_file, orient='records', indent=2)
        print(f"Exported {len(out_df)} records to {output_file}")
        return self

    def run(self, output_file='clean_movies.json'):
        """Run complete pipeline."""
        print("=== Running Data Validation Pipeline ===")
        self.load().profile().clean().validate().check_quality().export(output_file)
        print("=== Pipeline Complete ===")
        return self

# Test
pipeline = DataValidationPipeline('messy_movies.json')
pipeline.run('clean_movies.json')


---

# Part 8: Challenge Problems

### Challenge 8.1: Fuzzy Duplicate Detection

Sometimes duplicates have slight variations (e.g., "The Matrix" vs "Matrix, The").

Write a function that finds potential duplicates using fuzzy string matching.

**Hint**: Use the `fuzzywuzzy` or `rapidfuzz` library.

In [ ]:
# YOUR CODE HERE
def fuzzy_deduplicate(df, title_col='Title', threshold=0.85):
    """
    Detect near-duplicate rows using fuzzy string matching on titles.
    Returns a DataFrame with a 'fuzzy_duplicate_of' column.
    """
    from difflib import SequenceMatcher
    titles = df[title_col].str.lower().str.strip().tolist()
    n = len(titles)
    duplicate_of = [None] * n

    for i in range(n):
        if duplicate_of[i] is not None:
            continue
        for j in range(i + 1, n):
            if duplicate_of[j] is not None:
                continue
            ratio = SequenceMatcher(None, titles[i], titles[j]).ratio()
            if ratio >= threshold:
                duplicate_of[j] = titles[i]

    df = df.copy()
    df['fuzzy_duplicate_of'] = duplicate_of
    fuzzy_dups = df[df['fuzzy_duplicate_of'].notna()]
    print(f"Found {len(fuzzy_dups)} fuzzy duplicates (threshold={threshold})")
    if not fuzzy_dups.empty:
        print(fuzzy_dups[[title_col, 'fuzzy_duplicate_of']].to_string(index=False))
    return df

# Test
fuzzy_test = pd.DataFrame({
    'Title': ['The Matrix', 'Matrix, The', 'Inception', 'Incpeption', 'Avatar'],
    'Year':  [1999, 1999, 2010, 2010, 2009]
})
result = fuzzy_deduplicate(fuzzy_test, threshold=0.8)


### Challenge 8.2: Automatic Type Inference

Write a function that automatically infers the correct data type for each column by analyzing the values.

In [ ]:
# YOUR CODE HERE
def infer_and_cast_types(df):
    """
    Automatically infer the correct data type for each column
    and cast where possible.
    """
    import re
    df = df.copy()
    type_report = {}

    for col in df.columns:
        sample = df[col].dropna().astype(str)

        # Try integer (year, votes)
        int_pct = sample.str.match(r'^\d+$').mean()
        # Try float (ratings)
        float_pct = sample.str.match(r'^\d*\.?\d+$').mean()
        # Detect currency strings
        currency_pct = sample.str.match(r'^\$[\d,]+$').mean()
        # Detect "N min" runtime strings
        runtime_pct = sample.str.match(r'^\d+ min$').mean()

        if currency_pct > 0.5:
            df[col] = df[col].apply(clean_box_office)
            type_report[col] = 'currency → int'
        elif runtime_pct > 0.5:
            df[col] = df[col].apply(clean_runtime)
            type_report[col] = 'runtime → int'
        elif int_pct > 0.8:
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')
            type_report[col] = 'string → Int64'
        elif float_pct > 0.8:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            type_report[col] = 'string → float'
        else:
            type_report[col] = 'kept as string'

    print("Type Inference Report:")
    for col, t in type_report.items():
        print(f"  {col:<20}: {t}")
    return df

# Test
df_inferred = infer_and_cast_types(df_messy)
print("\nInferred dtypes:")
print(df_inferred.dtypes)


---

# Summary

In this lab, you learned:

1. **CLI Inspection**: Using jq and Unix tools for quick data exploration
2. **Data Profiling**: Systematic analysis of data quality issues
3. **Data Cleaning**: Converting messy strings to proper types
4. **Schema Validation**: Using Pydantic for type safety
5. **Quality Checks**: Automated assertions for data quality

## Next Week

**Week 3: Data Labeling & Annotation**
- Setting up Label Studio
- Annotation workflows
- Measuring inter-annotator agreement